# GK2A AMI SW038 다음 프레임 예측 (ConvLSTM)

2분 간격 위성영상 30장(15:00~15:58)을 이용한 **next-frame prediction** 파이프라인.

1. 데이터 적재 · 정규화 · Persistence 베이스라인
2. 패치 슬라이딩 윈도우 데이터셋 생성
3. ConvLSTM2D(**잔차 + SSIM 손실**) 학습 · 예측 · 시각화

> 데이터가 1시간짜리 단일 시퀀스라 **연습/PoC 용도**다. 패치로 잘라 샘플 수를 늘려 학습하지만,
> 일반화 성능보다는 파이프라인 검증이 목적이다.
>
> **실행 전 참고**
> - (A) M1 GPU 가속을 쓰려면 `tensorflow-metal` 설치 필요. (C) 입력을 900->300 으로 다운샘플해 속도를 크게 높였다.
> - 더 빠르게 확인만 하려면 `EPOCHS`를 5 정도로 낮춰라. 원본 해상도로 돌리려면 `TARGET = None`.
> - Persistence(다음=현재) 베이스라인이 매우 강하다(2분 사이 구름이 거의 안 움직임).
>   30장짜리 데이터로 from-scratch ConvLSTM이 이를 이기긴 어렵다 — 수치가 베이스라인 근처면 정상이다.

In [ ]:
import glob, os, re
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

%matplotlib inline
plt.close('all')

DATA_DIR = '/Users/eunbumkim/Documents/practice/data/NetCDF'
VAR = 'image_pixel_values'

IN_FRAMES = 4     # 입력 시퀀스 길이 (과거 N장)
TARGET = 300      # (C) 입력 다운샘플 해상도. 900->300 (정수배 3). None 이면 원본 900 유지
PATCH = 96        # 패치 한 변 크기 (TARGET=300 에 맞춰 축소)
STRIDE = 96       # 패치 간격 (작게 하면 겹쳐서 샘플 증가)
FILTERS = 16      # ConvLSTM 필터 수 (키우면 표현력↑ 속도↓)
EPOCHS = 20       # 빠른 확인은 5로 낮춰라
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)
print('tf', tf.__version__)

In [ ]:
# (A) M1 GPU(Metal) 확인 + mixed precision 활성화
#  - GPU가 '없음'이면:  !pip install tensorflow-metal==1.1.0  후 커널 재시작
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else '없음 (CPU) — tensorflow-metal 설치 필요')

# M1은 compute capability<7.0 이라 mixed precision 효과가 제한적일 수 있다.
# 불안정하거나 도움이 안 되면 아래 한 줄을 주석 처리(float32)해도 된다.
keras.mixed_precision.set_global_policy('mixed_float16')
print('precision policy:', keras.mixed_precision.global_policy().name)

In [ ]:
# 시간순 정렬해 모든 프레임을 (T, H, W) 로 적재
def list_nc_files(data_dir):
  files = glob.glob(os.path.join(data_dir, 'gk2a*sw038*.nc'))
  def ts(path):
    m = re.search(r'(\d{12})\.nc$', path)
    return m.group(1) if m else ''
  return sorted(files, key=ts)

files = list_nc_files(DATA_DIR)
print(f'파일 수: {len(files)}')
print('처음   :', os.path.basename(files[0]))
print('마지막 :', os.path.basename(files[-1]))

frames = []
for f in files:
  with xr.open_dataset(f) as ds:
    frames.append(ds[VAR].values.astype(np.float32))
frames = np.stack(frames, axis=0)
print('원본 frames:', frames.shape, frames.dtype)

# (C) 다운샘플: 900 -> TARGET, 정수배 평균 풀링(area)으로 구름 구조 보존
def downsample(arr, target):
  T, H, W = arr.shape
  fy, fx = H // target, W // target
  arr = arr[:, :target * fy, :target * fx]
  return arr.reshape(T, target, fy, target, fx).mean(axis=(2, 4)).astype(np.float32)

if TARGET and TARGET < frames.shape[1]:
  frames = downsample(frames, TARGET)
  print('다운샘플 후 :', frames.shape)

In [ ]:
# 전체 30장 기준 global min/max 로 0~1 정규화
GMIN, GMAX = float(frames.min()), float(frames.max())
print(f'global min={GMIN}, max={GMAX}')

def normalize(x):   return (x - GMIN) / (GMAX - GMIN)
def denormalize(x): return x * (GMAX - GMIN) + GMIN

frames_n = normalize(frames)   # (T, H, W) in [0, 1]

idxs = np.linspace(0, len(frames_n) - 1, 6).astype(int)
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, i in zip(axes, idxs):
  ax.imshow(frames_n[i], cmap='gray')
  ax.set_title(f't={i}')
  ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Persistence 베이스라인: '다음 프레임 = 현재 프레임' 가정의 오차
def ssim_metric(a, b):
  a = a[..., None].astype(np.float32)   # (N, H, W, 1)
  b = b[..., None].astype(np.float32)
  return float(tf.reduce_mean(tf.image.ssim(a, b, max_val=1.0)))

mae_list, ssim_list = [], []
for t in range(len(frames_n) - 1):
  cur, nxt = frames_n[t], frames_n[t + 1]
  mae_list.append(np.mean(np.abs(nxt - cur)))
  ssim_list.append(ssim_metric(cur[None], nxt[None]))

print(f'Persistence  MAE = {np.mean(mae_list):.5f}   SSIM = {np.mean(ssim_list):.4f}')
print('→ 구름 이동이 느려 베이스라인이 의외로 강하다. 모델은 이 수치를 이겨야 의미가 있다.')

In [ ]:
# 슬라이딩 윈도우 + 패치로 학습 샘플 생성
# 한 윈도우: frames[w : w+IN_FRAMES] 입력  ->  frames[w+IN_FRAMES] 타깃
def build_dataset(frames_n, t_start, t_end, in_frames=IN_FRAMES, patch=PATCH, stride=STRIDE):
  T, H, W = frames_n.shape
  ys = list(range(0, H - patch + 1, stride))
  xs = list(range(0, W - patch + 1, stride))
  X, Y = [], []
  for w in range(t_start, t_end - in_frames):
    seq = frames_n[w:w + in_frames]   # (in_frames, H, W)
    tgt = frames_n[w + in_frames]     # (H, W)
    for y in ys:
      for x in xs:
        X.append(seq[:, y:y + patch, x:x + patch])
        Y.append(tgt[y:y + patch, x:x + patch])
  X = np.array(X)[..., None].astype(np.float32)   # (N, in_frames, p, p, 1)
  Y = np.array(Y)[..., None].astype(np.float32)   # (N, p, p, 1)
  return X, Y

In [ ]:
# 시간 순서 유지 분할 (무작위 셔플 금지)
T = len(frames_n)
split = int(T * 0.8)   # 앞 80% 학습, 뒤 20% 검증

X_train, Y_train = build_dataset(frames_n, 0, split)
X_val,   Y_val   = build_dataset(frames_n, split - IN_FRAMES, T)

print('train:', X_train.shape, Y_train.shape)
print('val  :', X_val.shape, Y_val.shape)
# 주의: val 입력 일부가 train 구간과 겹칠 수 있으나, 예측 '타깃' 프레임은 분리되어 있다.

In [ ]:
# (2: 잔차 + SSIM 손실) 흐릿함 완화
#  - 모델은 '변화량 Δ'만 예측하고 마지막 입력 프레임에 더한다 -> 입력의 선명함을 계승
#    예측 = 마지막 입력 프레임 + Δ   (residual skip)
#  - 손실 = 0.5*MAE + 0.5*(1-SSIM)   (엣지/구조 보존 유도)
#  - 주의(A): Metal 에서 5D BatchNorm 비호환이라 BN 은 쓰지 않는다.

def ssim_mae_loss(y_true, y_pred):
  mae = tf.reduce_mean(tf.abs(y_true - y_pred))
  ssim = tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
  return 0.5 * mae + 0.5 * (1.0 - ssim)

def build_model(in_frames=IN_FRAMES, filters=FILTERS):
  # ── 왜 keras.Sequential 이 아니라 Functional API 인가? ───────────────────────
  # Sequential 은 '레이어 -> 레이어 -> ...' 로 한 줄로만 이어지는 구조다.
  # 이 모델은 마지막에 두 텐서를 '더한다':
  #     예측 = (입력의 마지막 프레임) + (신경망이 예측한 변화량 Δ)
  # 즉 입력에서 갈래(branch)가 나오고 끝에서 합쳐지는(merge) 구조라 일직선이 아니다.
  # 이런 잔차(residual) 구조는 Sequential 로 표현할 수 없어 Functional API 를 쓴다.
  #
  # ── layers.XXX(...)(텐서)  의 소괄호가 두 번 나오는 이유 ─────────────────────
  # Keras 레이어는 '함수처럼 호출할 수 있는 객체' 다. 두 단계로 나뉜다:
  #     conv = layers.ConvLSTM2D(...)   # (1) 레이어 '객체' 를 만든다 (설정만)
  #     x = conv(inp)                   # (2) 그 객체에 입력 텐서를 통과시킨다
  # 이 둘을 한 줄로 붙이면  layers.ConvLSTM2D(...)(inp)  가 되고,
  # 그래서 소괄호가 (생성)(호출) 두 번 나오는 것이다.
  # 아래에서는 이해를 돕기 위해 (1)과 (2)를 일부러 두 줄로 풀어 썼다.

  inp = keras.Input(shape=(in_frames, None, None, 1))   # 입력 텐서 (시간, 높이, 너비, 채널)

  lstm1 = layers.ConvLSTM2D(filters, (3, 3), padding='same',
                            return_sequences=True, activation='tanh')   # (1) 객체 생성
  x = lstm1(inp)                                                        # (2) 통과: 시퀀스 그대로 출력

  lstm2 = layers.ConvLSTM2D(filters, (3, 3), padding='same',
                            return_sequences=False, activation='tanh')
  x = lstm2(x)                                                          # 마지막 시점 하나만 출력

  to_delta = layers.Conv2D(1, (3, 3), padding='same',
                           activation=None, dtype='float32')           # 출력 1채널, 음수 가능하므로 linear
  delta = to_delta(x)                                                  # 변화량 Δ

  take_last = layers.Lambda(lambda t: t[:, -1], dtype='float32')       # 입력에서 마지막 프레임만 슬라이스
  last = take_last(inp)

  out = layers.Add(dtype='float32')([last, delta])                     # 예측 = 마지막 프레임 + Δ (잔차 합치기)

  model = keras.Model(inputs=inp, outputs=out)
  try:
    opt = keras.optimizers.legacy.Adam(1e-3)   # Apple Silicon 에서 더 빠름
  except Exception:
    opt = keras.optimizers.Adam(1e-3)
  model.compile(optimizer=opt, loss=ssim_mae_loss, metrics=['mae'])
  return model

model = build_model()
model.summary()

In [ ]:
callbacks = [
  keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
  keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=4, factor=0.5),
]
history = model.fit(
  X_train, Y_train,
  validation_data=(X_val, Y_val),
  epochs=EPOCHS, batch_size=16,
  callbacks=callbacks, verbose=1,
)

In [ ]:
# 학습 곡선
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('epoch'); plt.ylabel('loss = 0.5*MAE + 0.5*(1-SSIM)')
plt.legend(); plt.title('Training history')
plt.show()

# 검증셋에서 모델 vs Persistence 비교 (MAE 낮을수록 / SSIM 높을수록 좋음)
pred_val = np.clip(model.predict(X_val, batch_size=16, verbose=0), 0, 1)
model_mae  = float(np.mean(np.abs(pred_val - Y_val)))
pers_mae   = float(np.mean(np.abs(X_val[:, -1] - Y_val)))            # 예측=입력 마지막 프레임
model_ssim = ssim_metric(pred_val[..., 0], Y_val[..., 0])
pers_ssim  = ssim_metric(X_val[:, -1, ..., 0], Y_val[..., 0])

print(f'[Val] ConvLSTM    MAE = {model_mae:.5f}   SSIM = {model_ssim:.4f}')
print(f'[Val] Persistence MAE = {pers_mae:.5f}   SSIM = {pers_ssim:.4f}')
print('→ MAE 기준 모델이 베이스라인보다', '우수' if model_mae < pers_mae else '열등')

In [ ]:
# 전체 프레임 다음 시점 예측 (fully-conv 라 패치 학습 모델로 전체 추론 가능)
seq = frames_n[-IN_FRAMES - 1:-1][None, ..., None]   # (1, IN_FRAMES, H, W, 1)
true_next = frames_n[-1]
pred_next = np.clip(model.predict(seq, verbose=0)[0, ..., 0], 0, 1)   # 잔차합이 [0,1] 벗어날 수 있어 clip

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(frames_n[-2], cmap='gray'); axes[0].set_title('last input (t-1)')
axes[1].imshow(true_next, cmap='gray');    axes[1].set_title('ground truth (t)')
axes[2].imshow(pred_next, cmap='gray');    axes[2].set_title('prediction (t)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

print('full-frame MAE :', float(np.mean(np.abs(pred_next - true_next))))
print('full-frame SSIM:', ssim_metric(pred_next[None], true_next[None]))